# Step 07: Supervised fine-tuning (SFT)

Tests for `07-sft/sft.py`. Write code in the `.py` file, only import and test here.

The step 05 model only continues text. SFT keeps training it on **(prompt, response)** examples in one fixed format, so it learns that after the prompt comes an answer to it. New compared with pretraining: the loss is counted on the **response tokens only**.

Needs: the step 05 checkpoint, `uv run python 07-sft/prepare_sft.py` (data) and `uv run python 07-sft/sft.py` (training, about 15 minutes) for sections 5 onward.

Run the cells top to bottom (Run All).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import json
import numpy as np
import torch
import torch.nn.functional as F
import tiktoken

sys.path.append("07-sft")
from sft import (
    format_prompt, load_examples, make_batch, sample_batch, respond, word_use_rate,
    IGNORE, EOT_ID, BASE_CKPT,
)
sys.path.append("06-inference")
from inference import load_model

enc = tiktoken.get_encoding("gpt2")

## 1. The prompt format

The model needs a fixed layout to recognise a request. `### Response:` is the signal for "your turn". The same function builds the training prompts and the prompts you type later, so they always match.

In [2]:
print(format_prompt({"Words": "dragon, cookie", "Features": "Dialogue"}))
# ### Instruction:
# Write a short story.
# Words: dragon, cookie
# Features: Dialogue
#
# ### Response:

### Instruction:
Write a short story.
Words: dragon, cookie
Features: Dialogue

### Response:



## 2. The data

Each example is `prompt + story + end-of-text`, stored as token ids together with the prompt length. The model also sees the end-of-text token as a target, so it learns **when to stop**.

In [3]:
train_ex = load_examples("train")
val_ex = load_examples("val")
print(len(train_ex), len(val_ex))               # 20190 500

ids, prompt_len = train_ex[0]
print(ids.shape, prompt_len, ids[-1] == EOT_ID)  # a 1-D array, a prompt length, True
print(enc.decode(ids[:prompt_len]))              # the formatted prompt, ending with '### Response:'
print(enc.decode(ids[prompt_len:])[:200])        # the start of the story

20190 500
(236,) 31 True
### Instruction:
Write a short story.
Words: bury, wall, troubled
Features: BadEnding, Conflict

### Response:

Lily and Tom were playing in the garden. They liked to dig holes and bury things. They buried stones, leaves, and toys. They had fun.
But then they saw a big wall. The wall was new. It was in the way 


## 3. Masked targets

`make_batch` builds `x` and `y` of shape `(B, T)`, like pretraining, with two differences:
- examples have different lengths, so they are padded to the longest one
- targets that belong to the prompt, or to padding, are set to `-100`. `F.cross_entropy` skips those, so only the response counts.

A tiny hand-made batch (token ids are made up). Example A has 2 prompt tokens, B has 2 prompt tokens and is shorter.

In [4]:
A = (np.array([10, 11, 12, 13, EOT_ID]), 2)   # [P P R R EOT]
B = (np.array([20, 21, 22, EOT_ID]), 2)       # [P P R EOT]
x, y = make_batch([A, B], "cpu")
print(x.shape)    # torch.Size([2, 4])  (B=2, T=longest-1)
print(x)
# tensor([[   10,    11,    12,    13],
#         [   20,    21,    22, 50256]])      <- last 50256 in row 2 is padding
print(y)
# tensor([[ -100,    12,    13, 50256],       <- prompt target ignored, answer + EOT kept
#         [ -100,    22, 50256,  -100]])      <- last -100 is padding

torch.Size([2, 4])
tensor([[   10,    11,    12,    13],
        [   20,    21,    22, 50256]])
tensor([[ -100,    12,    13, 50256],
        [ -100,    22, 50256,  -100]])


## 4. Ignoring targets in the loss

Proof that `-100` targets really add nothing: the loss over 3 positions with one ignored equals the loss over the 2 kept positions.

In [5]:
torch.manual_seed(0)
logits = torch.randn(3, 5)                      # (positions, V=5)
targets = torch.tensor([IGNORE, 2, 1])

with_ignore = F.cross_entropy(logits, targets)
only_kept = F.cross_entropy(logits[1:], targets[1:])
print(torch.isclose(with_ignore, only_kept).item())   # True

True


In [6]:
np.random.seed(1)
x, y = sample_batch(train_ex, 4, "cpu")
print(x.shape, y.shape)                          # (4, T) both, T is at most 255

kept = y[0][y[0] != IGNORE].tolist()             # the tokens the loss is computed on, first example
print(enc.decode(kept)[:150])                    # starts with the story, no prompt text
print(f"{(y != IGNORE).float().mean():.0%} of target positions count")   # roughly 70-80%

torch.Size([4, 236]) torch.Size([4, 236])
Once upon a time, there was a little girl named Lily. She was very hungry and wanted to eat some cookies. But her mom said she couldn't have any until
72% of target positions count


## 5. Before SFT: the base model

Same prompt format, but the model has never seen it. Expect a story-like continuation that mostly ignores the requested words.

In [7]:
base, _ = load_model(BASE_CKPT)
val_prompts = json.load(open("07-sft/data/val_prompts.json"))

torch.manual_seed(0)
prompt = val_prompts[0]["prompt"]
print(prompt, "-> words asked:", val_prompts[0]["words"])
print(respond(base, prompt))
# a story-like text, mostly dialogue. In my run none of the asked words (visit, veil, lonely) appear

### Instruction:
Write a short story.
Words: visit, veil, lonely
Features: Twist
Summary: A lonely girl named Lily meets a friendly dog in the woods who turns out to be her grandma in disguise, bringing an end to her loneliness.

### Response:
 -> words asked: ['visit', 'veil', 'lonely']
One day, Lily and her dad went to the forest. They saw a big tree with a tree with lots of leaves and branches. Lily: "Look, dad, look at the tree!"
Tommy: "Wow, Lily! It looks so beautiful. I'm so glad it is like a new home."
Lily: "Thank you for helping me! You are the best friend. I love to hear the tree grow from nature."
Tommy: "You're welcome, Lily. I'm glad you like the tree. I love to play with you."
Lily: "Thank you, Tom. And I love you too. And I love the tree. It is very important. It is my favorite place to be. I am happy to see it."
Tom smiles. He says: "I love you too. I love you too, Lily. And I will never forget it. I hope you love you."


## 6. After SFT

Same prompt, now to the fine-tuned model. Expect a story that opens like one, uses some of the asked words, and stops by itself. Not perfect: a 30M model misses words and repeats names.

In [8]:
sft, _ = load_model("07-sft/out/sft_ckpt.pt")

torch.manual_seed(0)
print(respond(sft, prompt))
# In my run: opens with 'Once upon a time', uses 1 of the 3 asked words ('lonely'), repeats 'Lily' a lot,
# and stops by itself. Sampling is random, so your text will differ

Once upon a time, there was a lonely girl named Lily. She lived in a small house with her family. One day, Lily met a friendly girl named Lily.
"Hi Lily! Do you want to come with me?" asked Lily.
"Yes, please!" replied Lily.
Lily and Lily played together for a while. Suddenly, Lily realized her grandma was playing with a cat. "I want to play with you," said Lily.
"Sure, I can play with you," said Lily.
They played together for a while, but then Lily accidentally knocked over a cat. "Oh no, we need to go home now," said Lily.
"Don't worry, we'll go home," said Lily. "But we'll have to go home now."
Lily went home and saw her grandma waiting for her. "I'm sorry for you," said Lily. "We can come back soon." Lily smiled and hugged Lily.


## 7. Did it follow the request?

A simple, honest test: for 20 unseen prompts, what share of the requested words appear in the story? This measures only one part of "following the request", not story quality.

In [9]:
torch.manual_seed(0)
before = word_use_rate(base, val_prompts, 20, temperature=0.7, top_k=40)
after = word_use_rate(sft, val_prompts, 20, temperature=0.7, top_k=40)
print("base:", before)
print("sft: ", after)
# In my run (13 prompts that list words): base used about 21% of the asked words, SFT about 36%, nobody used ALL of them.
# The 40-prompt test inside sft.py gave 15% -> 39%, and all words in 0% -> 7% of stories. Small samples: expect noise

base: {'avg_share': 0.20512820512820512, 'all_words': 0.0, 'n': 13}
sft:  {'avg_share': 0.358974358974359, 'all_words': 0.0, 'n': 13}


## 8. Your own request

Change the words and features and run again. Try rare words, or many at once, and see where it breaks. The model only knows children's stories, so it cannot follow requests outside that world.

In [10]:
torch.manual_seed(1)
my_prompt = format_prompt({"Words": "dragon, cookie, brave", "Features": "Dialogue"})
print(respond(sft, my_prompt))

One day, a brave dragon named Tom was in a big castle. He had a big box with a big bow. Tom liked to play with the cake. He would make a big and loud sound.
One day, Tom saw a big box on the table. He wanted to see what was inside. He jumped on the box and opened the box. Inside the box, there was a big, dark dog in the room. The dog wanted to play with the dog.
Tom saw the dog and wanted to play too. He jumped on the box and ran to the box. He opened the box and found a big, shiny apple. Tom was very happy. They played with the cookie all day. They became friends and played together every day.
Summary: Tom, a brave dragon, and a brave dragon play with the toys in the living room.
